<a href="https://colab.research.google.com/github/nikhildhavale/pythonLearning/blob/main/lstmrnnregression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

RND = 42
np.random.seed(RND)
tf.random.set_seed(RND)

plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.titlesize"] = 14

print("TensorFlow:", tf.__version__)
print("All libraries loaded.")
def generate_adding_problem(n_samples, T, pos1_frac=0.1, pos2_frac=0.9, seed=None):
    if seed is not None:
        np.random.seed(seed)
    pos1 = int(T * pos1_frac)  # first marker index
    pos2 = int(T * pos2_frac)  # second marker index
    values = np.random.uniform(0, 1, size=(n_samples, T))
    markers = np.zeros((n_samples, T))
    markers[:, pos1] = 1
    markers[:, pos2] = 1
    X = np.stack([values, markers], axis=-1)  # (n_samples, T, 2)
    y = values[:, pos1] + values[:, pos2]  # (n_samples,)
    return X, y

T = 200
n_train, n_val, n_test = 12_000, 2_000, 2_000

X_train, y_train = generate_adding_problem(n_train, T, seed=RND)
X_val, y_val = generate_adding_problem(n_val, T, seed=RND + 1)
X_test, y_test = generate_adding_problem(n_test, T, seed=RND + 2)

print("Shapes:", X_train.shape, y_train.shape)
print("First sample: marker indices", np.where(X_train[0, :, 1] > 0.5)[0].tolist(), "-> target =", round(y_train[0], 4))
model_rnn = keras.Sequential([
    layers.Input(shape=(T, 2)),
    layers.SimpleRNN(64),
    layers.Dense(1),
])
model_rnn.compile(optimizer="adam", loss="mse", metrics=["mae"])

history_rnn = model_rnn.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=25,
    batch_size=64,
    verbose=1,
)

y_val_pred_rnn = model_rnn.predict(X_val).flatten()
print("SimpleRNN Val MSE:", round(mean_squared_error(y_val, y_val_pred_rnn), 6))
print("SimpleRNN Val R²:", round(r2_score(y_val, y_val_pred_rnn), 4))
plt.figure(figsize=(10, 4))
plt.plot(history_rnn.history["loss"], label="Train loss")
plt.plot(history_rnn.history["val_loss"], label="Val loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.title("SimpleRNN: training and validation loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
model_lstm = keras.Sequential([
    layers.Input(shape=(T, 2)),
    layers.LSTM(64),
    layers.Dense(1),
])
model_lstm.compile(optimizer="adam", loss="mse", metrics=["mae"])

history_lstm = model_lstm.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=25,
    batch_size=64,
    verbose=1,
)

y_val_pred_lstm = model_lstm.predict(X_val).flatten()
print("LSTM Val MSE:", round(mean_squared_error(y_val, y_val_pred_lstm), 6))
print("LSTM Val R²:", round(r2_score(y_val, y_val_pred_lstm), 4))
plt.figure(figsize=(10, 4))
plt.plot(history_lstm.history["loss"], label="Train loss")
plt.plot(history_lstm.history["val_loss"], label="Val loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.title("LSTM: training and validation loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
y_test_pred_rnn = model_rnn.predict(X_test).flatten()
y_test_pred_lstm = model_lstm.predict(X_test).flatten()

print("Test set comparison:")
print("  SimpleRNN  MSE =", round(mean_squared_error(y_test, y_test_pred_rnn), 6), "  R² =", round(r2_score(y_test, y_test_pred_rnn), 4))
print("  LSTM       MSE =", round(mean_squared_error(y_test, y_test_pred_lstm), 6), "  R² =", round(r2_score(y_test, y_test_pred_lstm), 4))
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(history_rnn.history["val_loss"], label="SimpleRNN")
axes[0].plot(history_lstm.history["val_loss"], label="LSTM")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Val MSE")
axes[0].legend()
axes[0].set_title("Validation loss: SimpleRNN vs LSTM")
axes[0].grid(True, alpha=0.3)

axes[1].scatter(y_test, y_test_pred_rnn, alpha=0.4, s=10, label="SimpleRNN")
axes[1].scatter(y_test, y_test_pred_lstm, alpha=0.4, s=10, label="LSTM")
axes[1].plot([0, 2], [0, 2], "k--", lw=1)
axes[1].set_xlabel("True sum")
axes[1].set_ylabel("Predicted sum")
axes[1].legend()
axes[1].set_title("Predicted vs true (test set)")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

TensorFlow: 2.19.0
All libraries loaded.
Shapes: (12000, 200, 2) (12000,)
First sample: marker indices [20, 180] -> target = 0.9529
Epoch 1/25
188/188 ━━━━━━━━━━━━━━━━━━━━ 8s 37ms/step - loss: 0.1467 - mae: 0.3116 - val_loss: 0.0868 - val_mae: 0.2488
Epoch 2/25
188/188 ━━━━━━━━━━━━━━━━━━━━ 8s 43ms/step - loss: 0.0943 - mae: 0.2616 - val_loss: 0.0855 - val_mae: 0.2476
Epoch 3/25
188/188 ━━━━━━━━━━━━━━━━━━━━ 8s 44ms/step - loss: 0.0924 - mae: 0.2598 - val_loss: 0.0848 - val_mae: 0.2471
Epoch 4/25
188/188 ━━━━━━━━━━━━━━━━━━━━ 9s 36ms/step - loss: 0.0914 - mae: 0.2585 - val_loss: 0.0844 - val_mae: 0.2468
Epoch 5/25
188/188 ━━━━━━━━━━━━━━━━━━━━ 8s 42ms/step - loss: 0.0908 - mae: 0.2577 - val_loss: 0.0841 - val_mae: 0.2466
Epoch 6/25
187/188 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: 0.0894 - mae: 0.2562